# HW17. DBSCAN и плотностная кластеризация

Здесь я сравниваю DBSCAN с K-Means на нелинейных данных и на датасете Wine. Главная идея: K-Means режет пространство примерно выпуклыми областями, а DBSCAN ищет плотные группы и может отдельно помечать шум.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import load_wine, make_circles, make_moons
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Нелинейные датасеты

In [ ]:
X_moons, y_moons = make_moons(n_samples=500, noise=0.07, random_state=RANDOM_STATE)
X_circles, y_circles = make_circles(n_samples=500, noise=0.045, factor=0.45, random_state=RANDOM_STATE)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, s=18, cmap="tab10")
axes[0].set_title("make_moons: реальные группы")
axes[1].scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, s=18, cmap="tab10")
axes[1].set_title("make_circles: реальные группы")
plt.tight_layout()
plt.show()

## 2. K-Means как базовое сравнение

In [ ]:
km_moons = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=10)
km_circles = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=10)
labels_km_moons = km_moons.fit_predict(X_moons)
labels_km_circles = km_circles.fit_predict(X_circles)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_km_moons, s=18, cmap="tab10")
axes[0].set_title("K-Means на moons")
axes[1].scatter(X_circles[:, 0], X_circles[:, 1], c=labels_km_circles, s=18, cmap="tab10")
axes[1].set_title("K-Means на circles")
plt.tight_layout()
plt.show()

K-Means здесь ожидаемо ошибается: он хорошо работает с компактными облаками, но плохо описывает дуги и кольца.

## 3. k-distance plot и выбор `eps`

In [ ]:
def plot_k_distance(X, k, title):
    nn = NearestNeighbors(n_neighbors=k)
    distances, _ = nn.fit(X).kneighbors(X)
    kth_distance = np.sort(distances[:, -1])[::-1]
    plt.figure(figsize=(7, 4))
    plt.plot(kth_distance)
    plt.xlabel("Объекты, отсортированные по расстоянию")
    plt.ylabel(f"Расстояние до {k}-го соседа")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    return kth_distance

X_moons_scaled = StandardScaler().fit_transform(X_moons)
X_circles_scaled = StandardScaler().fit_transform(X_circles)

kdist_moons = plot_k_distance(X_moons_scaled, 5, "k-distance для moons")
kdist_circles = plot_k_distance(X_circles_scaled, 5, "k-distance для circles")

По графикам беру умеренные значения `eps`: достаточно большие, чтобы дуги не распались, но не настолько большие, чтобы всё слиплось в один кластер.

In [ ]:
eps_moons = 0.23
eps_circles = 0.20

labels_db_moons = DBSCAN(eps=eps_moons, min_samples=5).fit_predict(X_moons_scaled)
labels_db_circles = DBSCAN(eps=eps_circles, min_samples=5).fit_predict(X_circles_scaled)

def cluster_stats(labels):
    return {
        "clusters": len(set(labels)) - (1 if -1 in labels else 0),
        "noise": int((labels == -1).sum()),
    }

print("moons:", cluster_stats(labels_db_moons))
print("circles:", cluster_stats(labels_db_circles))

In [ ]:
def plot_clusters(X, labels, title, ax):
    noise = labels == -1
    ax.scatter(X[~noise, 0], X[~noise, 1], c=labels[~noise], s=18, cmap="tab10", alpha=0.85)
    ax.scatter(X[noise, 0], X[noise, 1], c="black", marker="x", s=35, label="noise")
    ax.set_title(title)
    if noise.any():
        ax.legend()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_clusters(X_moons_scaled, labels_db_moons, "DBSCAN на moons", axes[0])
plot_clusters(X_circles_scaled, labels_db_circles, "DBSCAN на circles", axes[1])
plt.tight_layout()
plt.show()

## 4. Силуэт: K-Means и DBSCAN

In [ ]:
def safe_silhouette(X, labels):
    mask = labels != -1
    unique = set(labels[mask])
    if len(unique) < 2:
        return np.nan
    return silhouette_score(X[mask], labels[mask])

score_km_moons = silhouette_score(X_moons_scaled, KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=10).fit_predict(X_moons_scaled))
score_db_moons = safe_silhouette(X_moons_scaled, labels_db_moons)

print(f"K-Means silhouette на moons: {score_km_moons:.3f}")
print(f"DBSCAN silhouette на moons без шума: {score_db_moons:.3f}")

Силуэт не всегда честно награждает DBSCAN на сложных формах: метрика любит выпуклые и хорошо разделённые облака. Поэтому графики здесь важнее одной цифры.

## 5. Влияние `eps`

In [ ]:
eps_values = [0.06, 0.12, 0.18, 0.23, 0.35, 0.55]
fig, axes = plt.subplots(2, 3, figsize=(12, 7))

for eps, ax in zip(eps_values, axes.ravel()):
    labels = DBSCAN(eps=eps, min_samples=5).fit_predict(X_moons_scaled)
    stats = cluster_stats(labels)
    plot_clusters(X_moons_scaled, labels, f"eps={eps}\nclusters={stats['clusters']}, noise={stats['noise']}", ax)

plt.tight_layout()
plt.show()

При маленьком `eps` почти всё становится шумом. При слишком большом `eps` разные области могут объединиться. Нормальное значение находится около точки, где кластеры уже устойчивые, но ещё не сливаются.

## 6. DBSCAN на Wine

In [ ]:
wine = load_wine()
X_wine = pd.DataFrame(wine.data, columns=wine.feature_names)
y_wine = wine.target
X_wine_scaled = StandardScaler().fit_transform(X_wine)

_ = plot_k_distance(X_wine_scaled, 4, "k-distance для Wine")

eps_wine = 2.5
min_samples_wine = 4
labels_wine = DBSCAN(eps=eps_wine, min_samples=min_samples_wine).fit_predict(X_wine_scaled)
wine_stats = cluster_stats(labels_wine)
noise_ratio = (labels_wine == -1).mean()

print("Число кластеров:", wine_stats["clusters"])
print("Шумовых точек:", wine_stats["noise"])
print(f"Доля шума: {noise_ratio:.2%}")
print("Silhouette DBSCAN:", safe_silhouette(X_wine_scaled, labels_wine))

In [ ]:
noise_mask = labels_wine == -1
noise_profile = X_wine.loc[noise_mask].mean()
overall_profile = X_wine.mean()
profile_diff = ((noise_profile - overall_profile) / X_wine.std()).abs().sort_values(ascending=False)

display(pd.DataFrame({
    "noise_mean": noise_profile,
    "overall_mean": overall_profile,
    "abs_standardized_diff": profile_diff,
}).sort_values("abs_standardized_diff", ascending=False).head(8))

## 7. Сравнение с K-Means на Wine

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_wine_2d = pca.fit_transform(X_wine_scaled)

labels_km_wine = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10).fit_predict(X_wine_scaled)
score_km_wine = silhouette_score(X_wine_scaled, labels_km_wine)
score_db_wine = safe_silhouette(X_wine_scaled, labels_wine)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_clusters(X_wine_2d, labels_wine, f"DBSCAN, silhouette={score_db_wine:.3f}", axes[0])
axes[1].scatter(X_wine_2d[:, 0], X_wine_2d[:, 1], c=labels_km_wine, s=30, cmap="tab10")
axes[1].set_title(f"K-Means, silhouette={score_km_wine:.3f}")
plt.tight_layout()
plt.show()

## Итог

DBSCAN лучше K-Means, когда кластеры имеют сложную форму и есть шумовые объекты. Но на многомерных данных с разной плотностью он становится чувствительным к масштабу признаков и выбору `eps`. Для Wine K-Means часто выглядит стабильнее, потому что классы ближе к компактным группам, а DBSCAN часть объектов считает шумом.